In [29]:
import requests
#Daily index for Monday 21 Sep 2026
URL = "https://www.sec.gov/Archives/edgar/daily-index/2026/QTR3/master.20260921.idx"
Result = requests.get(URL , headers ={"User-Agent": "Keval kevals25fx@gmail.com"})
print(type(Result))
# Result methods are .text , .content , .json , .status_code , .headers
print(Result.text)
print(Result.status_code)
print(Result.headers)
print(Result.content)



<class 'requests.models.Response'>
Description:           Daily Index of EDGAR Dissemination Feed
Last Data Received:    Sep 21, 2026
Comments:              webmaster@sec.gov
Anonymous FTP:         ftp://ftp.sec.gov/edgar/
 
CIK|Company Name|Form Type|Date Filed|File Name
--------------------------------------------------------------------------------
1000275|ROYAL BANK OF CANADA|424B2|20260921|edgar/data/1000275/0000950103-26-014237.txt
1000275|ROYAL BANK OF CANADA|424B2|20260921|edgar/data/1000275/0000950103-26-014239.txt
1000275|ROYAL BANK OF CANADA|424B2|20260921|edgar/data/1000275/0000950103-26-014241.txt
1000275|ROYAL BANK OF CANADA|424B2|20260921|edgar/data/1000275/0000950103-26-014244.txt
1000275|ROYAL BANK OF CANADA|424B2|20260921|edgar/data/1000275/0000950103-26-014246.txt
1000275|ROYAL BANK OF CANADA|424B2|20260921|edgar/data/1000275/0000950103-26-014247.txt
1000275|ROYAL BANK OF CANADA|424B2|20260921|edgar/data/1000275/0000950103-26-014248.txt
1000275|ROYAL BANK OF CANADA|4

In [30]:
from datetime import date

def index_url(day: date) -> str:
    quarter = (day.month - 1) // 3 + 1
    return f"https://www.sec.gov/Archives/edgar/daily-index/{day.year}/QTR{quarter}/master.{day.strftime('%Y%m%d')}.idx"

In [31]:
index_url(date(2026, 9, 21))

'https://www.sec.gov/Archives/edgar/daily-index/2026/QTR3/master.20260921.idx'

In [32]:
import time
useragentheader = {"User-Agent": "Keval kevals25fx@gmail.com"}
def fetch (url : str) -> bytes :
    time.sleep(0.2) # To avoid rate limiting by SEC
    result = requests.get(url , headers = useragentheader , timeout = 30)
    if result.status_code != 200 :
        raise RuntimeError(f"Request to {url} failed with status code {result.status_code}")
    return result.content




In [33]:
fetch(index_url(date(2026, 9, 21)))

b"Description:           Daily Index of EDGAR Dissemination Feed\nLast Data Received:    Sep 21, 2026\nComments:              webmaster@sec.gov\nAnonymous FTP:         ftp://ftp.sec.gov/edgar/\n \nCIK|Company Name|Form Type|Date Filed|File Name\n--------------------------------------------------------------------------------\n1000275|ROYAL BANK OF CANADA|424B2|20260921|edgar/data/1000275/0000950103-26-014237.txt\n1000275|ROYAL BANK OF CANADA|424B2|20260921|edgar/data/1000275/0000950103-26-014239.txt\n1000275|ROYAL BANK OF CANADA|424B2|20260921|edgar/data/1000275/0000950103-26-014241.txt\n1000275|ROYAL BANK OF CANADA|424B2|20260921|edgar/data/1000275/0000950103-26-014244.txt\n1000275|ROYAL BANK OF CANADA|424B2|20260921|edgar/data/1000275/0000950103-26-014246.txt\n1000275|ROYAL BANK OF CANADA|424B2|20260921|edgar/data/1000275/0000950103-26-014247.txt\n1000275|ROYAL BANK OF CANADA|424B2|20260921|edgar/data/1000275/0000950103-26-014248.txt\n1000275|ROYAL BANK OF CANADA|424B2|20260921|edgar

In [34]:
FORM_4 = {"4", "4/A"}
def form4_filenames(index: bytes) -> list[str]:
    
    lines = index.decode("latin-1").splitlines()
    start = next(i for i, line in enumerate(lines) if line.startswith("---")) + 1
    rows = [line.split("|") for line in lines[start:]]
    return sorted({filename for cik, name, form, filed, filename in rows if form in FORM_4})

In [35]:
index = fetch(index_url(date(2026, 9, 21)))
filenames = form4_filenames(index)
len(filenames), filenames[:3]

(817,
 ['edgar/data/1006655/0001452310-26-000007.txt',
  'edgar/data/1006655/0001833972-26-000006.txt',
  'edgar/data/1006655/0001957952-26-000006.txt'])

In [36]:
filing = fetch("https://www.sec.gov/Archives/" + filenames[0])
print("\\n".join(filing.decode("latin-1").splitlines()[:60]))

<SEC-DOCUMENT>0001452310-26-000007.txt : 20260921\n<SEC-HEADER>0001452310-26-000007.hdr.sgml : 20260921\n<ACCEPTANCE-DATETIME>20260921161545\nACCESSION NUMBER:		0001452310-26-000007\nCONFORMED SUBMISSION TYPE:	4\nPUBLIC DOCUMENT COUNT:		1\nCONFORMED PERIOD OF REPORT:	20260917\nFILED AS OF DATE:		20260921\nDATE AS OF CHANGE:		20260921\n\nREPORTING-OWNER:	\n\n	OWNER DATA:	\n		COMPANY CONFORMED NAME:			Loyd Kelly William\n		CENTRAL INDEX KEY:			0001452310\n		ORGANIZATION NAME:           	\n\n	FILING VALUES:\n		FORM TYPE:		4\n		SEC ACT:		1934 Act\n		SEC FILE NUMBER:	001-32942\n		FILM NUMBER:		261394119\n\n	MAIL ADDRESS:	\n		STREET 1:		10000 MEMORIAL DRIVE\n		STREET 2:		SUITE 550\n		CITY:			HOUSTON\n		STATE:			TX\n		ZIP:			77024\n\nISSUER:		\n\n	COMPANY DATA:	\n		COMPANY CONFORMED NAME:			EVOLUTION PETROLEUM CORP\n		CENTRAL INDEX KEY:			0001006655\n		STANDARD INDUSTRIAL CLASSIFICATION:	CRUDE PETROLEUM & NATURAL GAS [1311]\n		ORGANIZATION NAME:           	01 Energy & Transportation\n		EIN:		

In [ ]:
def extract()